# Enhancing Agents with Callbacks 

**Challenge 2**: Enhance Weather Agent with Callbacks 

## Features
- 🌦️ Real-time weather data from National Weather Service API
- 📍 Location geocoding with Google Maps API
- 🤖 Built with Google Agent Development Kit
- 🧪 Comprehensive testing for multiple US cities

## Step 1: Install Dependencies

In [1]:
# !pip install google-adk google-genai requests python-dotenv nest-asyncio -q
%pip install "google-adk[extensions]" litellm google-genai requests python-dotenv nest-asyncio -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Import Libraries

In [7]:
import os
import json
import requests
import asyncio
import uuid
from typing import Dict, Any, Optional

# Enable nested event loops for Jupyter
import nest_asyncio
nest_asyncio.apply()

# Google ADK imports
from google.adk.agents.llm_agent import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai.types import Content, Part
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse

import vertexai
from vertexai.preview import reasoning_engines

import logging
from typing import Any, Dict, Optional

from dotenv import load_dotenv
load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Step 3: Configuration

Set your API keys here (optional - notebook works without them for common cities)

In [3]:
# API Keys
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# Project configuration
PROJECT_ID = "qwiklabs-gcp-02-138827e82db5"
LOCATION = "us-central1"

# Set environment variables for Vertex AI / Google GenAI SDK
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Initialize Vertex AI globally
vertexai.init(project=PROJECT_ID, location=LOCATION)

print("✅ Configuration loaded and Vertex AI initialized")

✅ Configuration loaded and Vertex AI initialized


## Step 4: Tool 1 - National Weather Service API

Retrieves weather data using latitude and longitude coordinates.
Follows PEP 8 style with type hints and comprehensive docstrings.

In [4]:
def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
    """
    Retrieve current weather data from the National Weather Service API.
    
    Args:
        latitude: Latitude coordinate in decimal degrees (-90.0 to 90.0)
        longitude: Longitude coordinate in decimal degrees (-180.0 to 180.0)
    
    Returns:
        Dictionary with weather data:
        - status: 'success' or 'error'
        - temperature: Temperature in Fahrenheit
        - conditions: Weather conditions
        - wind_speed: Wind speed
        - location: Location name
    
    Example:
        >>> weather = get_weather_by_coordinates(37.7749, -122.4194)
        >>> print(weather['temperature'])
        62
    """
    try:
        # Get forecast grid endpoint
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        headers = {
            'User-Agent': 'WeatherAgent/1.0 (Educational)',
            'Accept': 'application/json'
        }
        
        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        points_data = points_response.json()
        
        # Get forecast
        forecast_url = points_data['properties']['forecast']
        forecast_response = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_response.raise_for_status()
        forecast_data = forecast_response.json()
        
        # Extract current period
        current = forecast_data['properties']['periods'][0]
        location = points_data['properties']['relativeLocation']['properties']
        
        return {
            'status': 'success',
            'location': f"{location['city']}, {location['state']}",
            'temperature': current['temperature'],
            'temperature_unit': current['temperatureUnit'],
            'conditions': current['shortForecast'],
            'detailed_forecast': current['detailedForecast'],
            'wind_speed': current['windSpeed'],
            'wind_direction': current['windDirection']
        }
        
    except Exception as e:
        return {'status': 'error', 'error': str(e)}

print("✅ Weather function created")

✅ Weather function created


## Step 5: Tool 2 - Google Maps Geocoding API

Converts location names to coordinates. Includes fallback for common cities.

In [5]:
def geocode_location(location: str, api_key: Optional[str] = None) -> Dict[str, Any]:
    """
    Convert location name to latitude and longitude coordinates.
    
    Args:
        location: City name or address to geocode
        api_key: Optional Google Maps API key
    
    Returns:
        Dictionary with coordinates:
        - status: 'success' or 'error'
        - latitude: Latitude coordinate
        - longitude: Longitude coordinate
        - formatted_address: Full address
    
    Example:
        >>> coords = geocode_location('San Francisco, CA')
        >>> print(coords['latitude'], coords['longitude'])
        37.7749 -122.4194
    """
    # Fallback coordinates for common US cities
    CITY_COORDS = {
        'san francisco, ca': {'lat': 37.7749, 'lng': -122.4194},
        'new york city, ny': {'lat': 40.7128, 'lng': -74.0060},
        'chicago, il': {'lat': 41.8781, 'lng': -87.6298},
        'miami, fl': {'lat': 25.7617, 'lng': -80.1918},
        'seattle, wa': {'lat': 47.6062, 'lng': -122.3321},
        'austin, tx': {'lat': 30.2672, 'lng': -97.7431},
        'los angeles, ca': {'lat': 34.0522, 'lng': -118.2437}
    }
    
    location_key = location.lower().strip()
    
    # Try fallback first
    if location_key in CITY_COORDS:
        coords = CITY_COORDS[location_key]
        return {
            'status': 'success',
            'latitude': coords['lat'],
            'longitude': coords['lng'],
            'formatted_address': location,
            'source': 'fallback'
        }
    
    # Try Google Maps API if key provided
    maps_key = api_key or GOOGLE_MAPS_API_KEY
    if maps_key:
        try:
            url = 'https://maps.googleapis.com/maps/api/geocode/json'
            response = requests.get(url, params={'address': location, 'key': maps_key}, timeout=10)
            data = response.json()
            
            if data['status'] == 'OK':
                result = data['results'][0]
                loc = result['geometry']['location']
                return {
                    'status': 'success',
                    'latitude': loc['lat'],
                    'longitude': loc['lng'],
                    'formatted_address': result['formatted_address'],
                    'source': 'google_maps'
                }
        except Exception as e:
            pass
    
    return {
        'status': 'error',
        'error': f'Location not found. Available cities: {list(CITY_COORDS.keys())}'
    }

print("✅ Geocoding function created")

✅ Geocoding function created


## Step 6: Test Individual Functions

In [6]:
# Test geocoding
print("Testing Geocoding:")
coords = geocode_location("San Francisco, CA")
print(json.dumps(coords, indent=2))

# Test weather
print("\nTesting Weather:")
if coords['status'] == 'success':
    weather = get_weather_by_coordinates(coords['latitude'], coords['longitude'])
    print(json.dumps(weather, indent=2))

Testing Geocoding:
{
  "status": "success",
  "latitude": 37.7749,
  "longitude": -122.4194,
  "formatted_address": "San Francisco, CA",
  "source": "fallback"
}

Testing Weather:
{
  "status": "success",
  "location": "San Francisco, CA",
  "temperature": 68,
  "temperature_unit": "F",
  "conditions": "Partly Sunny",
  "detailed_forecast": "Partly sunny. High near 68, with temperatures falling to around 66 in the afternoon. West wind 8 to 15 mph, with gusts as high as 22 mph.",
  "wind_speed": "8 to 15 mph",
  "wind_direction": "W"
}


## Step 7: Define Callback Functions & Logging

In [8]:
# Set up logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("WeatherAgentCallbacks")


# Mock/Helper moderation checker (Slide 3)
def check_user_input(text: str) -> str:
    """Checks input against moderation rules."""
    prohibited_terms = ["malicious", "exploit", "drop table", "hack"]
    if any(term in text.lower() for term in prohibited_terms):
        return "BAD"
    return "OK"


# 1. Before Model Callback - Logging (Slide 1)
def log_user_prompt(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Logs the user prompt before sending to the LLM."""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            logger.info(
                "[%s] USER » %s",
                callback_context.agent_name,
                last.parts[0].text.strip(),
            )
    return None  # Return None to allow processing to continue


# 2. Before Model Callback - Moderation (Slide 3)
def moderate_user_prompt(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Validates user input and blocks inappropriate prompts."""
    try:
        if llm_request.contents:
            last = llm_request.contents[-1]
            if last.parts and last.parts[0].text:
                user_text = last.parts[0].text.strip()
                result_text = check_user_input(user_text)

                if result_text.strip().upper() == "BAD":
                    return LlmResponse(
                        content={
                            "role": "model",
                            "parts": [
                                {
                                    "text": "Message violates our content guidelines."
                                }
                            ],
                        }
                    )
    except Exception as e:
        logger.exception("Moderation callback failed: %s", e)

    return None


# 3. Chained Before Callback (Slide 4)
def chained_before_callback(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Chains moderation and logging prior to LLM execution."""
    # 1. Moderation check
    moderation_result = moderate_user_prompt(callback_context, llm_request)
    if moderation_result is not None:
        return moderation_result  # STOP: message was inappropriate

    # 2. Log user input
    log_user_prompt(callback_context, llm_request)

    return None  # Allow agent to proceed


# 4. After Model Callback - Response Logging (Slide 2)
def log_model_response(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """Logs the model response after generation."""
    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            logger.info(
                "[%s] MODEL » %s", callback_context.agent_name, txt.strip()
            )

    return None


print("✅ Callbacks and moderation pipeline defined successfully!")

✅ Callbacks and moderation pipeline defined successfully!


## Step 8: Create ADK Agent with Tools

In [9]:
# 1. Choose model target
ACTIVE_MODEL = "gemini_flash"  # or "claude_sonnet"

# 2. Model Registry
MODEL_REGISTRY = {
    "gemini_flash": "gemini-3.1-pro-preview",
    "claude_sonnet": LiteLlm(model="anthropic/claude-3-5-sonnet"),
}

# 3. Create Agent with Callbacks
weather_agent = Agent(
    model=MODEL_REGISTRY[ACTIVE_MODEL],
    name="weather_assistant",
    description="Weather assistant providing real-time weather info for US locations",
    instruction="""You are a helpful weather assistant.

When users ask about weather:
1. Use geocode_location to convert city name to coordinates
2. Use get_weather_by_coordinates to get weather data
3. Provide a clear, friendly summary
4. Alert on extreme conditions (temp >95°F or <32°F, high winds >25mph)

Be concise and informative.""",
    tools=[geocode_location, get_weather_by_coordinates],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

# 4. Wrap into AdkApp for Vertex AI Reasoning Engines
app = reasoning_engines.AdkApp(
    agent=weather_agent,
    enable_tracing=False,
)

# 5. Local In-Memory Runner
runner = InMemoryRunner(
    agent=weather_agent, app_name=f"Weather Assistant ({ACTIVE_MODEL})"
)

print("✅ Agent with callbacks, AdkApp, and local runner created successfully")

✅ Agent with callbacks, AdkApp, and local runner created successfully


## Step 9: Create User Session

In [10]:
# Create session directly in the runner's session store
user_id = "test-user-id"
app_name = getattr(runner, "app_name", "weather_assistant")

session = await runner.session_service.create_session(
    app_name=app_name, user_id=user_id
)

session_id = session.id if hasattr(session, "id") else session.get("id")
print(f"✅ Runner session created successfully: {session_id}")

✅ Runner session created successfully: 8c8dc478-74f2-49ae-8a83-8207bf930756


## Step 10: Agent Execution Function

In [11]:
import nest_asyncio

nest_asyncio.apply()


async def ask_weather_agent(query: str, session_id: str = None) -> str:
    """Query the weather agent using runner.run_async."""
    try:
        user_id = "test-user-id"
        app_name = getattr(runner, "app_name", "weather_assistant")

        # Fallback: create session if none provided
        if not session_id:
            current_session = await runner.session_service.create_session(
                app_name=app_name, user_id=user_id
            )
            session_id = (
                current_session.id
                if hasattr(current_session, "id")
                else current_session.get("id")
            )

        content = Content(role="user", parts=[Part(text=query)])
        final_text = None

        async for event in runner.run_async(
            user_id=user_id, session_id=session_id, new_message=content
        ):
            if hasattr(event, "is_final_response") and event.is_final_response():
                if (
                    hasattr(event, "content")
                    and hasattr(event.content, "parts")
                    and event.content.parts
                ):
                    final_text = event.content.parts[0].text
            elif (
                hasattr(event, "content")
                and hasattr(event.content, "parts")
                and event.content.parts
            ):
                final_text = event.content.parts[0].text

        return final_text or "No response received"
    except Exception as e:
        return f"Error: {str(e)}"


def query_weather(query: str, session_id: str = None) -> str:
    """Synchronous wrapper for Jupyter notebooks."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(ask_weather_agent(query, session_id))


print("✅ Agent execution functions ready")

✅ Agent execution functions ready


## Step 11: Test Agent with Single Query

In [12]:
response = query_weather(
    "What's the weather in San Francisco?", session_id=session_id
)
print(response)

/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()
INFO:WeatherAgentCallbacks:[weather_assistant] USER » What's the weather in San Francisco?
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response r

The current weather in San Francisco, CA is partly sunny with a temperature of 68°F. Winds are blowing from the west at 8 to 15 mph, with occasional gusts up to 22 mph. Expect a pleasant day!


## Step 11: Test Multiple US Cities

In [13]:
# Test cities
TEST_CITIES = [
    "San Francisco, CA",
    "New York City, NY",
    "Chicago, IL",
    "Miami, FL",
    "Seattle, WA"
]

print("="*70)
print("WEATHER AGENT TEST SUITE")
print("="*70)

results = []
for city in TEST_CITIES:
    print(f"\n🌆 Testing: {city}")
    print("-"*70)
    
    query = f"What's the weather in {city}?"
    response = query_weather(query)
    
    print(response)
    results.append({'city': city, 'success': 'error' not in response.lower()})

# Summary
print("\n" + "="*70)
print("TEST SUMMARY")
print("="*70)
success_count = sum(1 for r in results if r['success'])
print(f"Total: {len(results)}")
print(f"Successful: {success_count}")
print(f"Failed: {len(results) - success_count}")
print(f"Success Rate: {(success_count/len(results)*100):.1f}%")

INFO:WeatherAgentCallbacks:[weather_assistant] USER » What's the weather in San Francisco, CA?
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False


WEATHER AGENT TEST SUITE

🌆 Testing: San Francisco, CA
----------------------------------------------------------------------


INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:WeatherAgentCallbacks:[weather_assistant] MODEL » The current weather in San Francisco, CA is **Partly Sunny** with a temperature of **68°F**. Winds are coming from the west at 8 to 15 mph (with gusts up to 22 mph).
INFO:WeatherAgentCallbacks:[weather_assistant] USER » What's the weather in New York City, NY?
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False


The current weather in San Francisco, CA is **Partly Sunny** with a temperature of **68°F**. Winds are coming from the west at 8 to 15 mph (with gusts up to 22 mph).

🌆 Testing: New York City, NY
----------------------------------------------------------------------


INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:WeatherAgentCallbacks:[weather_assistant] MODEL » The current weather in New York City is **80°F** and **Mostly Sunny**. The wind is coming from the northeast at 7 to 13 mph. It looks like a beautiful day!
INFO:WeatherAgentCallbacks:[weather_assistant] USER » What's the weather in Chicago, IL?
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False


...

Currently in Chicago, IL, the temperature is 82°F with showers and thunderstorms likely. Winds are coming from the south at 5 to 10 mph. 

Make sure to grab an umbrella if you're heading out!

🌆 Testing: Miami, FL
----------------------------------------------------------------------


INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:WeatherAgentCallbacks:[weather_assistant] MODEL » The current weather in Miami, FL is 90°F with a chance of showers and thunderstorms. The wind is blowing from the south at 7 to 10 mph. 

Note that it is feeling quite hot with heat index values reaching up to 106°F, so make sure to stay cool and hydrated!
INFO:WeatherAgentCallbacks:[weather_assistant] USER » What's the weather in Seattle, WA?
INFO:google_adk.google.adk.models.google_llm:Sending out request

The current weather in Miami, FL is 90°F with a chance of showers and thunderstorms. The wind is blowing from the south at 7 to 10 mph. 

Note that it is feeling quite hot with heat index values reaching up to 106°F, so make sure to stay cool and hydrated!

🌆 Testing: Seattle, WA
----------------------------------------------------------------------


INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:WeatherAgentCallbacks:[weather_assistant] MODEL » The current weather in Seattle, WA is mostly cloudy with a temperature of 81°F. Expect gentle northwest winds at 2 to 6 mph. It's a nice, warm day out there!


The current weather in Seattle, WA is mostly cloudy with a temperature of 81°F. Expect gentle northwest winds at 2 to 6 mph. It's a nice, warm day out there!

TEST SUMMARY
Total: 5
Successful: 5
Failed: 0
Success Rate: 100.0%
